In [7]:
import pandas as pd
import networkx as nx

In [8]:
df = pd.read_csv("../data/processed/transactions_processed.csv")
graph_metrics = pd.read_csv("../data/processed/graph_metrics.csv")

df.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,Hour,Day,Month,Weekend,Amount Difference,Currency Match,Same Bank,Time Difference,Large Transaction
0,2022-09-01 00:00:00,1422,8014C7B60,1110,8017559F0,689.98,US Dollar,689.98,US Dollar,Cash,0,0,Thursday,9,False,0.0,True,False,NaN,False
1,2022-09-01 00:00:00,10416,806DC4EC0,10416,806DC4EC0,3.85,US Dollar,3.85,US Dollar,Reinvestment,0,0,Thursday,9,False,0.0,True,True,0.0,False
2,2022-09-01 00:00:00,2368,806DB8570,2368,806DB8570,4847.38,US Dollar,4847.38,US Dollar,Reinvestment,0,0,Thursday,9,False,0.0,True,True,0.0,False
3,2022-09-01 00:00:00,394,800B02E20,394,800B02E20,12.11,US Dollar,12.11,US Dollar,Reinvestment,0,0,Thursday,9,False,0.0,True,True,0.0,False
4,2022-09-01 00:00:00,21566,806DB5D90,21566,806DB5D90,3.55,US Dollar,3.55,US Dollar,Reinvestment,0,0,Thursday,9,False,0.0,True,True,0.0,False


In [9]:
accounts = pd.concat([
    df["Account"],
    df["Account.1"]
]).unique()[:5000]

In [10]:
def calculate_account_risk(account_id):
    sent = df[df["Account"] == account_id]
    received = df[df["Account.1"] == account_id]

    risk_score = 0
    reasons = []

    total_sent = sent["Amount Paid"].sum()
    total_received = received["Amount Received"].sum()
    transaction_count = len(sent) + len(received)

    if transaction_count > 20:
        risk_score += 20
        reasons.append("High transaction frequency")

    if total_sent > 100000:
        risk_score += 20
        reasons.append("High total outgoing amount")

    if len(sent["Account.1"].unique()) > 10:
        risk_score += 20
        reasons.append("Money sent to many different accounts")

    if len(received["Account"].unique()) > 10:
        risk_score += 20
        reasons.append("Money received from many different accounts")

    if sent["Large Transaction"].sum() > 2:
        risk_score += 20
        reasons.append("Multiple large transactions detected")

    return min(risk_score, 100), reasons

In [11]:
sample_account = accounts[0]

score, reasons = calculate_account_risk(sample_account)

print("Account:", sample_account)
print("Risk Score:", score)
print("Reasons:", reasons)

Account: 8014C7B60
Risk Score: 20
Reasons: ['High total outgoing amount']


In [12]:
risk_results = []

for account in accounts:
    score, reasons = calculate_account_risk(account)

    risk_results.append({
        "Account": account,
        "Risk_Score": score,
        "Risk_Reasons": "; ".join(reasons)
    })

risk_df = pd.DataFrame(risk_results)

risk_df.head()

,Account,Risk_Score,Risk_Reasons
0,8014C7B60,20,High total outgoing amount
1,806DC4EC0,0,
2,806DB8570,0,
3,800B02E20,0,
4,806DB5D90,0,


In [13]:
def detect_smurfing(account_id):
    sent = df[df["Account"] == account_id]

    near_threshold = sent[
        (sent["Amount Paid"] >= 45000) &
        (sent["Amount Paid"] <= 50000)
    ]

    if len(near_threshold) >= 3:
        return True
    
    return False

In [14]:
def detect_mule_account(account_id):
    sent = df[df["Account"] == account_id]
    received = df[df["Account.1"] == account_id]

    if len(received) >= 5 and len(sent) >= 5:
        if sent["Amount Paid"].sum() >= received["Amount Received"].sum() * 0.8:
            return True

    return False

In [16]:
risk_df["Smurfing_Flag"] = risk_df["Account"].apply(detect_smurfing)
risk_df["Mule_Flag"] = risk_df["Account"].apply(detect_mule_account)

risk_df.head()

,Account,Risk_Score,Risk_Reasons,Smurfing_Flag,Mule_Flag
0,8014C7B60,20,High total outgoing amount,False,False
1,806DC4EC0,0,,False,False
2,806DB8570,0,,False,False
3,800B02E20,0,,False,False
4,806DB5D90,0,,False,False


In [17]:
laundering_accounts = set(
    df[df["Is Laundering"] == 1]["Account"]
).union(
    set(df[df["Is Laundering"] == 1]["Account.1"])
)

risk_df["Known_Laundering_Account"] = risk_df["Account"].isin(laundering_accounts)

risk_df.head()

,Account,Risk_Score,Risk_Reasons,Smurfing_Flag,Mule_Flag,Known_Laundering_Account
0,8014C7B60,20,High total outgoing amount,False,False,False
1,806DC4EC0,0,,False,False,False
2,806DB8570,0,,False,False,False
3,800B02E20,0,,False,False,False
4,806DB5D90,0,,False,False,False


In [19]:
def risk_level(score):
    if score >= 70:
        return "High"
    elif score >= 40:
        return "Medium"
    else:
        return "Low"

risk_df["Risk_Level"] = risk_df["Risk_Score"].apply(risk_level)

risk_df.head()

,Account,Risk_Score,Risk_Reasons,Smurfing_Flag,Mule_Flag,Known_Laundering_Account,Risk_Level
0,8014C7B60,20,High total outgoing amount,False,False,False,Low
1,806DC4EC0,0,,False,False,False,Low
2,806DB8570,0,,False,False,False,Low
3,800B02E20,0,,False,False,False,Low
4,806DB5D90,0,,False,False,False,Low


In [20]:
print(risk_df.columns)

Index(['Account', 'Risk_Score', 'Risk_Reasons', 'Smurfing_Flag', 'Mule_Flag',
       'Known_Laundering_Account', 'Risk_Level'],
      dtype='object')


In [21]:
risk_df["Suspicious_Flag"] = (
    (risk_df["Risk_Level"] == "High") |
    (risk_df["Smurfing_Flag"] == True) |
    (risk_df["Mule_Flag"] == True) |
    (risk_df["Known_Laundering_Account"] == True)
)

risk_df.head()

,Account,Risk_Score,Risk_Reasons,Smurfing_Flag,Mule_Flag,Known_Laundering_Account,Risk_Level,Suspicious_Flag
0,8014C7B60,20,High total outgoing amount,False,False,False,Low,False
1,806DC4EC0,0,,False,False,False,Low,False
2,806DB8570,0,,False,False,False,Low,False
3,800B02E20,0,,False,False,False,Low,False
4,806DB5D90,0,,False,False,False,Low,False


In [22]:
laundering_accounts = set(
    df[df["Is Laundering"] == 1]["Account"]
).union(
    set(df[df["Is Laundering"] == 1]["Account.1"])
)

risk_df["Known_Laundering_Account"] = risk_df["Account"].isin(laundering_accounts)

risk_df.head()

,Account,Risk_Score,Risk_Reasons,Smurfing_Flag,Mule_Flag,Known_Laundering_Account,Risk_Level,Suspicious_Flag
0,8014C7B60,20,High total outgoing amount,False,False,False,Low,False
1,806DC4EC0,0,,False,False,False,Low,False
2,806DB8570,0,,False,False,False,Low,False
3,800B02E20,0,,False,False,False,Low,False
4,806DB5D90,0,,False,False,False,Low,False


In [23]:
risk_df["Suspicious_Flag"] = (
    (risk_df["Risk_Level"] == "High") |
    (risk_df["Smurfing_Flag"] == True) |
    (risk_df["Mule_Flag"] == True) |
    (risk_df["Known_Laundering_Account"] == True)
)

risk_df.head()

,Account,Risk_Score,Risk_Reasons,Smurfing_Flag,Mule_Flag,Known_Laundering_Account,Risk_Level,Suspicious_Flag
0,8014C7B60,20,High total outgoing amount,False,False,False,Low,False
1,806DC4EC0,0,,False,False,False,Low,False
2,806DB8570,0,,False,False,False,Low,False
3,800B02E20,0,,False,False,False,Low,False
4,806DB5D90,0,,False,False,False,Low,False


In [24]:
risk_df.to_csv(
    "../data/processed/account_risk_scores.csv",
    index=False
)

risk_df.head()

,Account,Risk_Score,Risk_Reasons,Smurfing_Flag,Mule_Flag,Known_Laundering_Account,Risk_Level,Suspicious_Flag
0,8014C7B60,20,High total outgoing amount,False,False,False,Low,False
1,806DC4EC0,0,,False,False,False,Low,False
2,806DB8570,0,,False,False,False,Low,False
3,800B02E20,0,,False,False,False,Low,False
4,806DB5D90,0,,False,False,False,Low,False


In [25]:
risk_df["Risk_Level"].value_counts()

Risk_Level
Low       4995
Medium       4
High         1
Name: count, dtype: int64

In [26]:
risk_df["Suspicious_Flag"].value_counts()

Suspicious_Flag
False    4993
True        7
Name: count, dtype: int64